# Data Merging — Stage 3 Cleaning 04: Normalise

## Input
- The four cleaned aggregate tables from notebook 01 (`Data/Data_Collection/Final/Stage_3_Cleaning/agg_market_daily_means.parquet`, `agg_market_daily_full_moments.parquet`, `agg_market_monthly_means.parquet`, `agg_market_monthly_full_moments.parquet`)
- `Data/Data_Collection/Final/Stage_3_Cleaning/weekly_raw.parquet` from notebook 02
- Shared helper modules: `lib.normalise.robust_expanding_zscore`, `lib.normalise.shift_max_stat`, and `lib.review` (thresholds, drop rules, reporting helpers)

## Purpose
Computes a causal, expanding-window robust z-score for every feature across all five tables, then runs a structured suite of post-hoc diagnostics and drop rules on the *raw* values (measured within a fixed diagnostic window) to decide which features are unreliable enough to exclude before modelling. This is the notebook where the stationarity scan's findings (Stage 3 notebook 03a) are turned into concrete keep/drop decisions, plus several additional rules not covered by that scan (correlated duplicates, boundary/crisis-driven distortion, warmup degeneracy).

The notebook is organized as a sequence of cells, each building on the previous:

## Cell 1 — Z-Score Computation
For each of the five tables, calls `robust_expanding_zscore` on every feature column (excluding `date`, the relevant target column, and a small set of binary regime indicators that pass through un-z-scored: `vix_above_20`, `vix_above_30`, `curve_inverted_2y10y`, `curve_inverted_3m10y`, `credit_stress`).

- Z-scores are computed **causally** over the full 2004–2024 sample using an expanding window — no look-ahead.
- Diagnostics for the drop rules are measured only over a fixed window (`rv.DIAG_START`–`rv.DIAG_END`, described as "Split B training window — the earliest split") so that decisions aren't made using information from later splits.
- A separate NBER crisis window (`rv.CRISIS_START`–`rv.CRISIS_END`) is tracked so boundary-mass statistics can be reported both with and without crisis periods included.
- Each table's z-scored output is saved immediately to `Stage_4_Normalised/`, and a per-feature diagnostic report is accumulated across all tables into `zscore_diagnostic_report.csv`.

## Cell 2 — Structural, Rule 4, and Rule 5 Pre-Drops
Runs three categories of "free" drops on **raw values** within the diagnostic window, before the z-score-based rules get a chance to fire — the idea being these are cheap, obviously-justified exclusions that shrink the workload for the harder diagnostic-based rules:

- **Structural exclusions** (`rv.structural_drops`) — factors excluded by construction rather than by measurement (presumably things already flagged in earlier stages that need to be re-caught here per-table). After running, the notebook checks which names in `rv.STRUCTURAL` were *never* matched in any table — these are either already removed upstream or misspelled, and are printed as a diagnostic. Any names in `rv.PENDING_VERIFICATION` are flagged for manual confirmation before a final run.
- **Rule 4 — trending level with a `_mom`/`_yoy` twin**: drops a raw level factor if its `raw_shift_max > 1.5` *and* it has a differenced twin already present, since the twin is stationary and the level is redundant. Full audit trail saved to `rule4_level_twin_audit.csv`.
- **Rule 5 — near-duplicate factors**: drops one of a pair of factors whose *first differences* correlate at `|ρ| > 0.995`, using an alphabetical tiebreak to decide which survives. Structural and Rule 4 drops take precedence over Rule 5 (`extra_drops.setdefault`, not overwrite). Full audit trail saved to `rule5_duplicate_audit.csv`. An absence of any qualifying pairs is treated as a reportable result in its own right, not silently skipped.

## Cell 3 — Apply All Rules and Summarize
Calls `rv.apply_rules` to combine the z-score diagnostics (`master`) with the structural/Rule-4/Rule-5 pre-drops (`extra_drops`) into a single decision table, respecting the exemption list for the pass-through binary indicators.

- Saves the full decision table to `decisions_aggregate.csv` and the list of surviving (kept) features to `surviving_features_aggregate.csv`.
- Prints drop counts by rule and bucket (`MACRO - DAILY`, `MACRO - WEEKLY`, `MACRO - MONTHLY`), noting explicitly that a feature can fail multiple rules so rule columns won't sum to the total dropped count.
- Prints the same breakdown per individual table (feature count, drop count, keep count, percent dropped).
- Prints a "primary reason" table: for each dropped feature, the *first* rule to fire (in a fixed precedence order), cross-tabbed by bucket — this is the number that gets reported cleanly rather than the multi-rule overlap.

## Cell 4 — Detailed Drop Listing
For each bucket, prints every dropped feature grouped by which rule fired, showing the full diagnostic row (modal share, std of z excluding capped values, boundary-mass percentages with/without crisis exclusion, sigma-floor "rung" indicator, warmup degeneracy flag, observation count, max |z|, full-sample shift_max on the z-score, and the date the diagnostic window starts). This is the primary human-review artifact — every drop decision is traceable back to its exact numbers.

## Cell 5 — Three Validation Checks on the Drop Rules Themselves
These exist to confirm the drop rules aren't producing false positives or hiding their own mechanics:

1. **Check 1 — sigma-floor rung crosstab.** Cross-tabs bucket against `sigma_f_rung` for features failing the low-variance side of Rule 2. Rung 1 means an ordinary MAD estimate (variance genuinely collapsed); rungs 2/3/4 mean the robust scale hit a floor, which is a *different* failure mode worth distinguishing from genuine variance collapse.
2. **Check 2 — does the z-score cap explain the boundary mass?** For features failing the boundary-mass rule (`R6_boundary_mass`), checks whether their `n_capped` count (values hard-clipped at |z|>10) is near zero. Since the cap only acts above |z|=10 but the boundary-mass statistic counts |z|>5, and capping biases the running sigma low by roughly 6% per capped point (inflating *later* z-scores), this check confirms the rule isn't just measuring its own clipping mechanism circularly.
3. **Check 3 — how many features the NBER crisis exclusion rescued.** Computes, for every feature, whether it would have failed the boundary-mass threshold using the full-window statistic vs. the crisis-excluded statistic, and reports the count of features that only pass because crisis-period observations were excluded from the count. This number is explicitly called out as "the write-up number."

## Cell 6 — Full-Sample Drift Measurement (Reported, Not Acted On)
Deliberately placed **last**, and explicitly *not* a drop rule. Uses `shift_max_stat` (the same statistic from the stationarity scan, applied here to the *z-scores themselves* rather than raw values) to measure drift between the early and late portions of the full 2004–2024 sample, but only for features that already survived every other rule — after also removing any features covered by `union_drop_list.csv` (a union drop list, presumably compiled across daily/monthly/weekly duplicates in the next notebook — the cell warns if that file doesn't exist yet and drift is measured pre-union as a stopgap).

The reasoning for *not* using this to drop features: `shift_max` measured between early and late sample **is** drift between the train and test periods, and dropping a feature because of that would be selection on the test-period distribution — a look-ahead-adjacent methodological error. So this is reported as a stated limitation of the surviving feature set rather than corrected for.

Uses `base_factor_map` (not `base_factor` per-column) when checking against `union_drop_list.csv`, because `_spread` only strips to its base name when the matching `_cwmean` sibling is visible in the same column list — without the full set, a bare `X_spread` would fail to match its base factor in the drop list and survive orphaned while its sibling moments were removed.

Reports drift percentile summary by bucket, cumulative counts above several thresholds (1.0–3.0), and the 25 worst-drifting surviving features.

## Output
- `Data/Data_Collection/Final/Stage_4_Normalised/{5 table names}.parquet` — z-scored feature tables.
- `Data/Data_Collection/Final/Stage_4_Normalised/zscore_diagnostic_report.csv` — per-feature z-score diagnostics across all tables.
- `Data/Data_Collection/Final/Stage_4_Normalised/rule4_level_twin_audit.csv`
- `Data/Data_Collection/Final/Stage_4_Normalised/rule5_duplicate_audit.csv`
- `Data/Data_Collection/Final/Stage_4_Normalised/decisions_aggregate.csv` — full keep/drop decision table with every rule outcome per feature.
- `Data/Data_Collection/Final/Stage_4_Normalised/surviving_features_aggregate.csv`
- `Data/Data_Collection/Final/Stage_4_Normalised/drift_limitation_aggregate.csv` — full-sample drift on surviving features, reported as a limitation, not acted on.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append('../..')
from lib.normalise import robust_expanding_zscore
import lib.review as rv

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 400)

IN_DIR  = Path('../../../Data/Data_Collection/Final/Stage_3_Cleaning')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_4_Normalised')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# filename -> (min_dates, target_col, frequency, display bucket)
TABLES = {
    'agg_market_daily_means.parquet':         (252, 'target_daily_return',   'daily',   'MACRO - DAILY'),
    'agg_market_daily_full_moments.parquet':  (252, 'target_daily_return',   'daily',   'MACRO - DAILY'),
    'weekly_raw.parquet':                     ( 52, None,                    'weekly',  'MACRO - WEEKLY'),
    'agg_market_monthly_means.parquet':       ( 24, 'target_monthly_return', 'monthly', 'MACRO - MONTHLY'),
    'agg_market_monthly_full_moments.parquet':( 24, 'target_monthly_return', 'monthly', 'MACRO - MONTHLY'),
}

BUCKET_ORDER = ['MACRO - DAILY', 'MACRO - WEEKLY', 'MACRO - MONTHLY']

# Binary regime indicators pass through un-z-scored and are exempt from all rules.
SKIP_ZSCORE = {'vix_above_20', 'vix_above_30',
               'curve_inverted_2y10y', 'curve_inverted_3m10y', 'credit_stress'}

print('=' * 100)
print('STAGE 3 / 04 - AGGREGATE NORMALISATION')
print('=' * 100)
print(f'  z-scores computed over : ALL rows (2004-2024), expanding, causal')
print(f'  diagnostics measured on: {rv.DIAG_START} .. {rv.DIAG_END}  '
      f'(Split B training window - the earliest split)')
print(f'  crisis excluded from pct_gt5_ex_crisis: {rv.CRISIS_START} .. {rv.CRISIS_END} (NBER)')

reports, bucket_of, freq_of = [], {}, {}

for fname, (min_dates, target_col, freq, bucket) in TABLES.items():
    path = IN_DIR / fname
    if not path.exists():
        print(f'\n  !! {fname} not found - skipped')
        continue

    tag = fname.replace('.parquet', '')
    bucket_of[tag], freq_of[tag] = bucket, freq

    print(f'\n{"-" * 100}\n  {bucket}  |  {tag}  (min_dates={min_dates})')
    df = pd.read_parquet(path)

    meta = ['date'] + ([target_col] if target_col in df.columns else [])
    skip = set(meta) | SKIP_ZSCORE
    feature_cols = [c for c in df.columns if c not in skip]

    present_binaries = sorted(SKIP_ZSCORE & set(df.columns))
    if present_binaries:
        print(f'    pass-through (not z-scored): {present_binaries}')

    df_z, rep = robust_expanding_zscore(
        df=df, feature_cols=feature_cols, date_col='date',
        min_dates=min_dates,
        diag_start=rv.DIAG_START, diag_end=rv.DIAG_END,
        crisis_start=rv.CRISIS_START, crisis_end=rv.CRISIS_END,
        verbose=True,
    )

    rep.insert(0, 'table_source', tag)
    reports.append(rep)

    df_z.to_parquet(OUT_DIR / fname, index=False, engine='pyarrow')
    print(f'    saved -> {fname}')

master = pd.concat(reports, ignore_index=True)
master.to_csv(OUT_DIR / 'zscore_diagnostic_report.csv', index=False)

print(f'\n{"=" * 100}')
print(f'  diagnostic report: {len(master):,} features -> zscore_diagnostic_report.csv')
print(master.groupby('table_source').size().to_string())

STAGE 3 / 04 - AGGREGATE NORMALISATION
  z-scores computed over : ALL rows (2004-2024), expanding, causal
  diagnostics measured on: 2004-01-01 .. 2015-12-31  (Split B training window - the earliest split)
  crisis excluded from pct_gt5_ex_crisis: 2007-12-01 .. 2009-06-30 (NBER)

----------------------------------------------------------------------------------------------------
  MACRO - DAILY  |  agg_market_daily_means  (min_dates=252)
    pass-through (not z-scored): ['credit_stress', 'curve_inverted_2y10y', 'curve_inverted_3m10y', 'vix_above_20', 'vix_above_30']
    327 features, 5,234 dates, 5,234 rows (1.0 rows/date)
    z-scored over ALL rows; report measured over 2,971 rows [2004-01-01 .. 2015-12-31]
    crisis rows excluded from pct_gt5_ex_crisis: 397 [2007-12-01 .. 2009-06-30]
    sigma_f rung 2:    2 feature(s)  (P95/1.96, >50% ties)
    warm-up trimmed at +/-10 robust sigma: 172 values
    contributions capped at +/-10: 898   sigma_f refreshes: 20
    std_of_z_ex_capped: me

In [2]:
# Everything here is measured on RAW values inside the diagnostic window.
# Order matters: structural and correlation drops are free and shrink what the
# z-diagnostics have to adjudicate.

extra_drops = {}
r4_audits, r5_audits = [], []

print('=' * 100)
print('A. STRUCTURAL EXCLUSIONS  (from factor construction, not measurement)')
print('=' * 100)

for fname, (_, target_col, _, bucket) in TABLES.items():
    path = IN_DIR / fname
    if not path.exists():
        continue
    tag = fname.replace('.parquet', '')

    raw = pd.read_parquet(path).sort_values('date', kind='stable')
    raw = raw[(raw['date'] >= rv.DIAG_START) & (raw['date'] <= rv.DIAG_END)]

    skip = {'date', target_col} | SKIP_ZSCORE
    feature_cols = [c for c in raw.columns if c not in skip and c != target_col]

    # --- structural -----------------------------------------------------------
    s = rv.structural_drops(feature_cols)
    for f, v in s.items():
        extra_drops[(tag, f)] = v

    remaining = [c for c in feature_cols if c not in s]


    # --- rule 4 ---------------------------------------------------------------
    d4, a4 = rv.rule4(raw, remaining, date_col='date')
    for f, v in d4.items():
        extra_drops[(tag, f)] = v
    if len(a4):
        a4.insert(0, 'table_source', tag)
        r4_audits.append(a4)

    # --- rule 5 ---------------------------------------------------------------
    d5, a5 = rv.rule5(raw, remaining)
    for f, v in d5.items():
        extra_drops.setdefault((tag, f), v)   # structural/R4 take precedence
    if len(a5):
        a5.insert(0, 'table_source', tag)
        r5_audits.append(a5)

    print(f'  {bucket:<16} {tag:<34} structural {len(s):>4}   R4 {len(d4):>3}   R5 {len(d5):>3}')

# Which structural names were never found anywhere? Either already removed in
# Stage 1.5 / Stage 3_01, or misspelled here.
found = {rv.base_factor(f) for (_, f) in extra_drops}
missing = sorted(set(rv.STRUCTURAL) - found)
print(f'\n  structural names not present in any aggregate table ({len(missing)}):')
print('   ', missing if missing else '(none)')
if rv.PENDING_VERIFICATION:
    print('\n  ** VERIFY BEFORE FINAL RUN:', rv.PENDING_VERIFICATION, '**')

print('\n' + '=' * 100)
print('RULE 4  - trending level with a _mom / _yoy twin   (drop if raw shift_max > 1.5)')
print('=' * 100)
r4 = pd.concat(r4_audits, ignore_index=True) if r4_audits else pd.DataFrame()
if len(r4):
    print(r4.sort_values('raw_shift_max', ascending=False).to_string(index=False))
    print(f'\n  fired {int(r4["dropped"].sum())} of {len(r4)} candidates')
else:
    print('  no level/twin pairs found')

print('\n' + '=' * 100)
print('RULE 5  - |rho| > 0.995 on first differences   (alphabetical tiebreak)')
print('=' * 100)
r5 = pd.concat(r5_audits, ignore_index=True) if r5_audits else pd.DataFrame()
if len(r5):
    print(r5.to_string(index=False))
else:
    print('  no pairs above 0.995 - documents an absence, which is itself a result')

r4.to_csv(OUT_DIR / 'rule4_level_twin_audit.csv', index=False)
r5.to_csv(OUT_DIR / 'rule5_duplicate_audit.csv', index=False)

A. STRUCTURAL EXCLUSIONS  (from factor construction, not measurement)
  MACRO - DAILY    agg_market_daily_means             structural    9   R4   0   R5  15
  MACRO - DAILY    agg_market_daily_full_moments      structural   33   R4   0   R5  43
  MACRO - WEEKLY   weekly_raw                         structural    0   R4   0   R5   0
  MACRO - MONTHLY  agg_market_monthly_means           structural   22   R4   8   R5   4
  MACRO - MONTHLY  agg_market_monthly_full_moments    structural   50   R4   8   R5  14

  structural names not present in any aggregate table (0):
    (none)

RULE 4  - trending level with a _mom / _yoy twin   (drop if raw shift_max > 1.5)
                   table_source              feature                              twins  raw_shift_max   med_early       med_mid      med_late  monotone  threshold  dropped
       agg_market_monthly_means      consumer_credit                consumer_credit_mom       4.933775   2161.4000   2460.100000   3081.603700      True        1.5 

In [3]:
decisions = rv.apply_rules(master, extra_drops, bucket_of, freq_of,
                           exempt=SKIP_ZSCORE)
decisions.to_csv(OUT_DIR / 'decisions_aggregate.csv', index=False)

surv = decisions.loc[decisions['action'] == 'keep', ['table_source', 'feature']]
surv.to_csv(OUT_DIR / 'surviving_features_aggregate.csv', index=False)

print('=' * 100)
print('DROP COUNTS BY RULE AND BUCKET')
print('=' * 100)
print('A feature can fail several rules, so rule columns do not sum to n_dropped.\n')
print(rv.rule_summary(decisions, BUCKET_ORDER).to_string(index=False))

print('\n' + '-' * 100)
print('SAME BREAKDOWN BY INDIVIDUAL TABLE')
print('-' * 100)
per_table = (decisions.groupby(['bucket', 'table_source'])
             .agg(n_features=('feature', 'size'),
                  n_dropped=('action', lambda s: (s == 'drop').sum()),
                  n_kept=('action', lambda s: (s == 'keep').sum()))
             .reset_index())
per_table['pct_dropped'] = (per_table['n_dropped'] / per_table['n_features']).map('{:.1%}'.format)
print(per_table.to_string(index=False))

print('\n' + '-' * 100)
print('PRIMARY REASON (first rule in precedence order) - unique features per reason')
print('-' * 100)
prim = (decisions[decisions['action'] == 'drop']
        .groupby(['bucket', 'rule_fired']).size()
        .unstack(fill_value=0).reindex(BUCKET_ORDER))
prim.columns = [rv.RULE_LABEL.get(c, c) for c in prim.columns]
print(prim.T.to_string())

DROP COUNTS BY RULE AND BUCKET
A feature can fail several rules, so rule columns do not sum to n_dropped.

         bucket  n_features  n_dropped  n_kept  S_structural  R4_trending_level_twin  R5_duplicate_rho  R0_unassessable  R1_modal_share  R2_std_bounds  R3_warmup_degenerate  R6_boundary_mass
  MACRO - DAILY        1318        136    1182            42                       0                58                0               0             18                     0                23
 MACRO - WEEKLY          34          2      32             0                       0                 0                0               0              0                     0                 2
MACRO - MONTHLY        1372        149    1223            72                      16                18                0               5             43                     7                28
          TOTAL        2724        287    2437           114                      16                76                0          

In [5]:
SHOW = ['table_source', 'feature', 'rule_fired', 'all_rules',
        'modal_share', 'std_of_z_ex_capped', 'pct_gt5', 'pct_gt5_ex_crisis', 'n_gt5_ex_crisis',
        'sigma_f_rung', 'warmup_degenerate', 'n_obs_diag', 'max_abs_z',
        'shift_max_z', 'd_start', 'reason']

FMT = {'modal_share': '{:.3f}', 'std_of_z_ex_capped': '{:.3f}',
       'pct_gt5': '{:.3%}', 'pct_gt5_ex_crisis': '{:.3%}',
       'max_abs_z': '{:,.1f}', 'shift_max_z': '{:.2f}'}

for b in BUCKET_ORDER:
    d = decisions[(decisions['bucket'] == b) & (decisions['action'] == 'drop')]
    print('\n' + '=' * 130)
    print(f'DROPPED  |  {b}  |  {len(d)} features')
    print('=' * 130)
    if not len(d):
        print('  none')
        continue
    for rule in rv.RULE_ORDER:
        dr = d[d['rule_fired'] == rule]
        if not len(dr):
            continue
        print(f'\n  --- {rv.RULE_LABEL[rule]}   ({len(dr)}) ---')
        print(dr[SHOW].sort_values('feature').to_string(index=False,
              formatters={k: v.format for k, v in FMT.items()}))


DROPPED  |  MACRO - DAILY  |  136 features

  --- Structural (construction)   (42) ---
                 table_source                      feature   rule_fired                       all_rules modal_share std_of_z_ex_capped pct_gt5 pct_gt5_ex_crisis  n_gt5_ex_crisis  sigma_f_rung  warmup_degenerate  n_obs_diag max_abs_z shift_max_z    d_start                                                                                                         reason
       agg_market_daily_means                 close_vs_mid S_structural                    S_structural       0.000              1.130  1.361%            0.689%               16             1              False        2718      26.7        0.08 2005-03-15                                                            [zero_denominator] Same construction as open_vs_mid
agg_market_daily_full_moments          close_vs_mid_cwkurt S_structural                    S_structural       0.000              0.922  0.736%            0.517%               12 

In [6]:


print('=' * 100)
print('CHECK 1 - sigma_f rung crosstab on the LOW side of rule 2')
print('=' * 100)
print('Rung 1 = ordinary MAD, so variance genuinely fell.')
print('Rungs 2/3/4 = floor-dominated, a different failure worth naming separately.\n')
low = decisions[decisions['std_of_z_ex_capped'] < rv.STD_LO]
print(pd.crosstab(low['bucket'], low['sigma_f_rung'], margins=True).to_string())

print('\n' + '=' * 100)
print('CHECK 2 - does the cap explain the boundary mass?')
print('=' * 100)
print('The cap acts at |z|>10; pct_gt5 counts |z|>5, so the 5-10 band was never')
print('capped. Capping still biases the running sigma low (~6% per capped point),')
print('which inflates later z. If n_capped is near zero for these features, the')
print('circularity is not driving the rule.\n')
r6 = decisions[decisions['all_rules'].str.contains('R6_boundary_mass', regex=False)]
if len(r6):
    print(r6[['bucket', 'feature', 'pct_gt5', 'pct_gt5_ex_crisis', 'n_gt5_ex_crisis',
              'n_capped', 'n_obs_diag', 'max_abs_z', 'std_of_z_ex_capped']]
          .sort_values('pct_gt5_ex_crisis', ascending=False)
          .head(40).to_string(index=False))
    print(f'\n  n_capped == 0 for {int((r6["n_capped"] == 0).sum())} of {len(r6)} '
          f'features failing R6')
else:
    print('  no features failed R6')

print('\n' + '=' * 100)
print('CHECK 3 - how many features the crisis exclusion rescued')
print('=' * 100)
n_all = (decisions['pct_gt5'] * decisions['n_obs_diag']).round()
would = (decisions['pct_gt5'] > rv.PCT_GT5_MAX) & (n_all >= rv.PCT_GT5_MIN_COUNT)
does  = (decisions['pct_gt5_ex_crisis'] > rv.PCT_GT5_MAX) & \
        (decisions['n_gt5_ex_crisis'] >= rv.PCT_GT5_MIN_COUNT)
resc  = decisions[would & ~does]
print(f'  fail on all-window pct_gt5   : {int(would.sum())}')
print(f'  fail ex-crisis (the rule)    : {int(does.sum())}')
print(f'  RESCUED by the NBER exclusion: {len(resc)}   <- write-up number')
if len(resc):
    print()
    print(resc[['bucket', 'feature', 'pct_gt5', 'pct_gt5_ex_crisis', 'max_abs_z']]
          .sort_values('pct_gt5', ascending=False).head(30).to_string(index=False))

CHECK 1 - sigma_f rung crosstab on the LOW side of rule 2
Rung 1 = ordinary MAD, so variance genuinely fell.
Rungs 2/3/4 = floor-dominated, a different failure worth naming separately.

sigma_f_rung      1  4  All
bucket                     
MACRO - DAILY    18  0   18
MACRO - MONTHLY  31  4   35
All              49  4   53

CHECK 2 - does the cap explain the boundary mass?
The cap acts at |z|>10; pct_gt5 counts |z|>5, so the 5-10 band was never
capped. Capping still biases the running sigma low (~6% per capped point),
which inflates later z. If n_capped is near zero for these features, the
circularity is not driving the rule.

         bucket                         feature  pct_gt5  pct_gt5_ex_crisis  n_gt5_ex_crisis  n_capped  n_obs_diag    max_abs_z  std_of_z_ex_capped
MACRO - MONTHLY                 ConvDebt_spread 0.222222           0.265306               26        22         117 9.000000e+07            1.699692
MACRO - MONTHLY                      ExchSwitch 0.102564           0

In [7]:
# Deliberately last. Drift is not a drop rule -- shift_max between early and
# late sample is drift between train and test, and dropping on it is selection
# on test-period distribution. It is measured here, on the FULL sample, only
# for features that already survived, and reported as a limitation.

from lib.normalise import shift_max_stat

union_path = OUT_DIR / 'union_drop_list.csv'
union_drop = (set(pd.read_csv(union_path)['base_factor'])
              if union_path.exists() else set())
if not union_drop:
    print('!! union_drop_list.csv not found - drift measured PRE-union.')
    print('   Run notebook 05 cell 6 first, then re-run this cell.\n')

rows = []
for fname, (_, target_col, _, bucket) in TABLES.items():
    path = OUT_DIR / fname
    if not path.exists():
        continue
    tag = fname.replace('.parquet', '')

    surviving = surv.loc[surv['table_source'] == tag, 'feature'].tolist()
    if not surviving:
        continue

    # base_factor_map, not base_factor per column: union_drop_list.csv holds
    # BASE factors, and _spread only strips when the matching _cwmean is
    # visible. Without the full column list X_spread maps to itself, is not
    # found in the list, and survives while its four siblings are removed.
    z = pd.read_parquet(path)
    bmap = rv.base_factor_map(z.columns)
    keep_cols = [c for c in surviving if bmap[c] not in union_drop]
    if not keep_cols:
        continue

    years = pd.DatetimeIndex(z['date']).year.to_numpy()
    for c in keep_cols:
        rows.append({'bucket': bucket, 'table_source': tag, 'feature': c,
                     'shift_max_z_full_sample':
                         shift_max_stat(z[c].to_numpy(dtype=float), years)})
    del z

drift = pd.DataFrame(rows)
drift.to_csv(OUT_DIR / 'drift_limitation_aggregate.csv', index=False)

print('=' * 100)
print('DRIFT ON SURVIVING FEATURES - full sample 2004-2024, REPORTED NOT ACTED ON')
print('=' * 100)
print(drift.groupby('bucket')['shift_max_z_full_sample']
      .describe(percentiles=[.5, .75, .9, .95, .99]).to_string())

for thr in (1.0, 1.5, 2.0, 2.5, 3.0):
    n = int((drift['shift_max_z_full_sample'] > thr).sum())
    print(f'  > {thr:<4} {n:>5}  ({n / len(drift):.1%})')

print('\n  worst 25:')
print(drift.nlargest(25, 'shift_max_z_full_sample').to_string(index=False))

DRIFT ON SURVIVING FEATURES - full sample 2004-2024, REPORTED NOT ACTED ON
                  count      mean       std       min       50%       75%       90%       95%       99%       max
bucket                                                                                                           
MACRO - DAILY    1103.0  0.819804  0.747330  0.003319  0.673853  1.201653  1.734501  2.052886  3.525962  5.447680
MACRO - MONTHLY  1206.0  0.983777  0.660441  0.016637  0.885907  1.301237  1.880905  2.271145  2.859821  4.761084
MACRO - WEEKLY     32.0  0.753400  0.550498  0.027019  0.576492  1.341762  1.493010  1.719964  1.754955  1.758041
  > 1.0    894  (38.2%)
  > 1.5    385  (16.4%)
  > 2.0    162  (6.9%)
  > 2.5     67  (2.9%)
  > 3.0     29  (1.2%)

  worst 25:
         bucket                    table_source                              feature  shift_max_z_full_sample
  MACRO - DAILY          agg_market_daily_means                               aa_oas                 5.447680
  MAC